## Helper Functions

In [ ]:
import json
import textwrap
from pathlib import Path
from time import perf_counter

from kilin.conversations import ConversationStore
from kilin.rag import RAGSession


def _print_wrapped(label, value, width=79):
    """Print text with a fixed display width, preserving paragraphs."""
    print(f"{label}:")
    for line in str(value).splitlines() or [""]:
        if not line.strip():
            print()
        else:
            print(textwrap.fill(
                line, width=width, break_long_words=False,
                break_on_hyphens=False,
            ))
    print("")


def create_conversation(project_dir):
    """Create an independent conversation and its active RAG session.

    Parameters
    ----------
    project_dir : str or Path
        Existing KíLín project directory.

    Returns
    -------
    tuple
        Conversation store, conversation record, and active session.
        Close the session after the last question.

    """
    store = ConversationStore(Path(project_dir))
    conversation = store.create()
    path = store.state_path(conversation)
    session = RAGSession(Path(project_dir), conversation_path=path)
    print(f"Conversation: {path}")
    return store, conversation, session


def ask_question(session, question):
    """Ask the active model and print a width-limited exchange.

    Parameters
    ----------
    session : RAGSession
        Active conversation session.
    question : str
        User question.

    Returns
    -------
    str
        Assistant response.

    """
    print("\n" + "=" * 79)
    _print_wrapped("USER", question)
    start = perf_counter()
    try:
        answer = session.ask(question)
    except Exception:
        print(f"Elapsed before failure: {perf_counter() - start:.3f} s")
        raise
    _print_wrapped("ASSISTANT", answer)
    print(f"Wall time: {perf_counter() - start:.3f} s")
    return answer


def inspect_conversation(conversation_path, turn_number=None):
    """Display routing, retrieval, model calls, tools, and trace timings.

    Parameters
    ----------
    conversation_path : str or Path
        Native KíLín trace JSON file.
    turn_number : int, optional
        One-based turn index, or None for all turns.

    Returns
    -------
    dict
        Unmodified native trace record.

    """
    with Path(conversation_path).open(encoding="utf-8") as file:
        trace = json.load(file)

    turns = list(trace["sessions"]["05_conversations"]["turns"].items())
    if turn_number is not None:
        if not 1 <= turn_number <= len(turns):
            raise ValueError("Invalid turn number.")
        turns = [turns[turn_number - 1]]

    def operation(stages, stage_name, operation_name):
        return (stages.get(stage_name, {})
                .get("operations", {}).get(operation_name, {}))

    def show_fields(heading, data, keys):
        print(f"\n{heading}")
        for key in keys:
            if key in data:
                _print_wrapped(f"  {key}", data[key])

    for turn_id, turn in turns:
        stages = turn.get("stages", {})
        print("\n" + "=" * 79)
        print(f"TURN: {turn_id}")
        print("=" * 79)

        question = operation(stages, "01_question", "01_turn")
        response = operation(
            stages, "01_question", "09_conversation_turn_representation"
        )
        show_fields("EXCHANGE", question.get("details", {}), ["question"])
        show_fields("ANSWER", response.get("details", {}), ["summary"])

        decision = operation(
            stages, "01_question", "05_intent_classification_decided"
        ).get("details", {})
        show_fields("INTENT", decision.get("classification", {}), [
            "intent", "standalone", "refers_back", "related_turn_ids"
        ])
        show_fields("ROUTING", decision, ["shortcut", "fallback"])
        show_fields("RESOLVED POLICY", decision.get("resolved", {}), [
            "history_policy", "retrieval_mode", "retrieval_strategy"
        ])
        settings = operation(
            stages, "01_question", "02_turn_settings_snapshot"
        ).get("details", {})
        show_fields("EFFECTIVE SETTINGS", settings, [
            "project_type_id", "backend", "chat_model", "router_model",
            "thinking", "temperature", "max_new_tokens", "retrieval_count"
        ])
        retrieval = operation(
            stages, "03_retrieval", "02_retrieval_result"
        )
        show_fields("RETRIEVAL", retrieval.get("details", {}), [
            "mode", "strategy", "evidence_count", "duration_ms"
        ])

        calls = []
        for stage_name, stage in stages.items():
            for name, entry in stage.get("operations", {}).items():
                if name == "01_llm_call_input" or name == "02_llm_call_output":
                    calls.append((stage_name, name, entry))
                elif "tool" in name.lower():
                    print(f"\nTOOL OPERATION: {stage_name}/{name}")
                    show_fields("  DETAILS", entry.get("details", {}), [
                        "tool", "tool_name", "status", "duration_ms"
                    ])
        for stage_name, name, entry in calls:
            details = entry.get("details", {})
            show_fields(f"MODEL CALL: {stage_name}/{name}", details, [
                "call_id", "model_id", "backend", "device", "thinking",
                "input_token_count", "prompt_token_count",
                "output_token_count", "finish_reason", "reply_duration_ms"
            ])
            show_fields("GENERATION SETTINGS", details.get("generation", {}), [
                "temperature", "max_new_tokens", "thinking"
            ])

        print("\nOPERATION TIMINGS (ms)")
        for stage_name, stage in stages.items():
            for name, entry in stage.get("operations", {}).items():
                timing = entry.get("timing", {})
                duration = timing.get("operation_duration_ms")
                if duration is not None:
                    print(f"  {stage_name}/{name}: {duration:.3f}")

    return trace


## Create Empty Literature Project

In [ ]:
from pathlib import Path

import kilin

# Locate the repository independently of the notebook working directory.
repo_root = Path.cwd().resolve()
while not (repo_root / "pyproject.toml").is_file():
    if repo_root == repo_root.parent:
        raise RuntimeError("Run this notebook from within the KíLín repository.")
    repo_root = repo_root.parent

project_dir = repo_root / "sandbox" / "trial_literature"
project_dir.parent.mkdir(parents=True, exist_ok=True)

# Do not overwrite a project created by an earlier run.
if project_dir.exists():
    raise FileExistsError(f"Project directory already exists: {project_dir}")

project = kilin.initialize_project(
    project_dir,
    kilin.PROJECT_TYPE_ID_LITERATURE,
)

print(f"Created literature project: {project_dir}")

## First Conversation without Sources

In [ ]:
store, conversation, session = create_conversation(project_dir)


In [ ]:
answer = ask_question(session, "hello")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=1,
)

In [ ]:
answer = ask_question(session, "What can you help me with?")

trace = inspect_conversation(
    store.state_path(conversation),
    turn_number=2,
)

In [ ]:
session.close()

## Add Articles to Project